<a href="https://colab.research.google.com/github/Likhithluck/Capstone-Project/blob/likhith/Converting_to_CSV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!sudo apt update
!sudo apt install -y tesseract-ocr

!pip install datasets pytesseract

Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:8 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [1,383 kB]
Hit:9 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [2,783 kB]
Get:13 http://security.ubuntu.com/ubuntu j

In [5]:
from datasets import load_dataset
import pytesseract
import json
import pandas as pd
from PIL import Image
import re
from datetime import datetime


# Load the dataset
ds = load_dataset("mychen76/invoices-and-receipts_ocr_v1", split="train")

class InvoiceParser:
    def __init__(self, text):
        self.text = text

    def extract_invoice_number(self):
        match = re.search(r"Invoice no:\s*(\d+)", self.text)
        return match.group(1) if match else None

    def extract_date(self):
        date_patterns = [
            r'\b(\d{1,2}/\d{1,2}/\d{4})\b',
            r'\b(\d{4}-\d{1,2}-\d{1,2})\b'
        ]

        for pattern in date_patterns:
            match = re.search(pattern, self.text)
            if match:
                raw_date = match.group(1)
                for fmt in ("%m/%d/%Y", "%d/%m/%Y", "%Y-%m-%d"):
                    try:
                        return datetime.strptime(raw_date, fmt).strftime("%Y-%m-%d")
                    except ValueError:
                        continue
        return None

    def extract_details(self):
        lines = self.text.split("\n")

        seller_name, seller_address, seller_tax, seller_iban = "", "", "", ""
        client_name, client_address, client_tax = "", "", ""

        seller_section, client_section = False, False
        seller_lines, client_lines = [], []

        for line in lines:
            line = line.strip()
            if line.startswith("Seller:"):
                seller_section = True
                client_section = False
                continue
            elif line.startswith("Client:"):
                seller_section = False
                client_section = True
                continue
            if seller_section:
                seller_lines.append(line)
            elif client_section:
                client_lines.append(line)

        if seller_lines:
            seller_name = seller_lines[0] if seller_lines[1] == "" else seller_lines[1]
            seller_address = " ".join(seller_lines[2:4])
            for line in seller_lines:
                if "Tax Id:" in line:
                    seller_tax = line.split(":")[1].strip()
                if "IBAN:" in line:
                    seller_iban = line.split(":")[1].strip()

        if client_lines:
            client_name = client_lines[0] if client_lines[1] == "" else client_lines[1]
            client_address = " ".join(client_lines[2:4])
            for line in client_lines:
                if "Tax Id:" in line:
                    client_tax = line.split(":")[1].strip()

        return {
            "Seller": {
                "Name": seller_name,
                "Address": seller_address,
                "Tax ID": seller_tax,
                "IBAN": seller_iban
            },
            "Client": {
                "Name": client_name,
                "Address": client_address,
                "Tax ID": client_tax
            }
        }

    def extract_summary(self):
        text = self.text
        summary_text = text.split("Total")[-1]

        vat_percent_match = re.search(r'VAT\s*\[\s*%\s*\]\s*(\d+%)', text)
        vat_percent = vat_percent_match.group(1) if vat_percent_match else None

        money_matches = re.findall(r'\$ ?\d{1,3}(?:[.,]\d{2})', summary_text)
        money_matches_cleaned = [match.replace(' ', '') for match in money_matches]
        money_values_sorted = sorted(
            money_matches_cleaned,
            key=lambda x: float(x.replace('$', '').replace(',', '.')),
            reverse=True
        )

        gross_worth = money_values_sorted[0] if len(money_values_sorted) > 0 else None
        net_worth = money_values_sorted[1] if len(money_values_sorted) > 1 else None
        vat_value  = money_values_sorted[2] if len(money_values_sorted) > 2 else None

        return {
            "VAT [%]": vat_percent,
            "Net worth": net_worth,
            "VAT": vat_value,
            "Gross worth": gross_worth
        }

    def extract_all_info(self):
        return {
            "Invoice Number": self.extract_invoice_number(),
            "Date of Issue": self.extract_date(),
            "Seller and Client Details": self.extract_details(),
            "Summary": self.extract_summary()
        }


# Function to check if OCR text fits standard invoice format
def is_standard_invoice(text):
    return all([
        "Invoice no:" in text,
        "Seller:" in text,
        "Client:" in text,
        "Total" in text,
    ])

# To store results
results = []

# Loop through the dataset
for i, item in enumerate(ds):
    image = item["image"]  # PIL Image
    ocr_text = pytesseract.image_to_string(image)

    if is_standard_invoice(ocr_text):
        # Use your parser on this text
        parser = InvoiceParser(ocr_text)
        structured = parser.extract_all_info()

        # Append to results list
        results.append({
            "ocr_text": ocr_text,
            "json_text": json.dumps(structured)
        })

        print(f"[✓] Parsed item {i}")

    else:
        print(f"[ ] Skipped item {i} - non-standard format")

# Save results to CSV
df = pd.DataFrame(results)
df.to_csv("parsed_invoices.csv", index=False)

print(f"Saved {len(results)} parsed invoices to 'parsed_invoices.csv'")


[✓] Parsed item 0
[✓] Parsed item 1
[✓] Parsed item 2
[✓] Parsed item 3
[✓] Parsed item 4
[✓] Parsed item 5
[✓] Parsed item 6
[✓] Parsed item 7
[✓] Parsed item 8
[✓] Parsed item 9
[✓] Parsed item 10
[✓] Parsed item 11
[✓] Parsed item 12
[✓] Parsed item 13
[✓] Parsed item 14
[✓] Parsed item 15
[✓] Parsed item 16
[✓] Parsed item 17
[✓] Parsed item 18
[✓] Parsed item 19
[✓] Parsed item 20
[✓] Parsed item 21
[✓] Parsed item 22
[✓] Parsed item 23
[✓] Parsed item 24
[✓] Parsed item 25
[✓] Parsed item 26
[✓] Parsed item 27
[✓] Parsed item 28
[✓] Parsed item 29
[✓] Parsed item 30
[✓] Parsed item 31
[✓] Parsed item 32
[✓] Parsed item 33
[✓] Parsed item 34
[✓] Parsed item 35
[✓] Parsed item 36
[✓] Parsed item 37
[✓] Parsed item 38
[✓] Parsed item 39
[✓] Parsed item 40
[✓] Parsed item 41
[✓] Parsed item 42
[✓] Parsed item 43
[✓] Parsed item 44
[✓] Parsed item 45
[✓] Parsed item 46
[✓] Parsed item 47
[✓] Parsed item 48
[✓] Parsed item 49
[✓] Parsed item 50
[✓] Parsed item 51
[✓] Parsed item 52
[✓]

In [6]:
df.to_csv("parsed_invoices.csv", index=False)

In [7]:
from google.colab import files
files.download('parsed_invoices.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>